# RAG Chatbot for Full Stack Academy (Gradio + OpenRouter + FAISS)
This notebook builds a simple RAG chatbot using website + CSV data.

In [26]:
!pip install langchain-openai

In [27]:
!pip install langchain faiss-cpu pandas requests beautifulsoup4 openai gradio tiktoken

In [28]:
!pip install langchain-text-splitters

!pip install langchain-google-genai


In [29]:
!pip install langchain-community faiss-cpu langchain-huggingface

In [30]:
import pandas as pd
import requests ## webscrapping
from bs4 import BeautifulSoup
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

import gradio as gr

## Enter OpenRouter API Key

In [31]:
OPENROUTER_API_KEY = "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"

os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

## Website Scraper

In [32]:
response = requests.get("https://fullstackacademy.in/")
soup = BeautifulSoup(response.text, "html.parser")
result = soup.get_text(separator=" ")

In [33]:
result

"\n \n \n \n \n \n \n Full Stack Academy | Gateway To Software Industry \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Full Stack Academy | Gateway To Software Industry \n \n \n \n \n \n \n   \n \n \n \n \n \n \n \n Menu \n \n   \n \n \n Home \n Courses \n Branches \n \n FSA – Tolichowki \n FSA – Gachibowli \n FSA – Ameerpet \n FSA – Charminar \n \n \n About \n Events \n Placements \n \n Companies \n Successfully Placed Candidates \n Candidates available for Placements \n \n \n Contact \n \n \n \n \n \n \n \n \n \n \n \n \n \n Get A Call \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n\t\t\t\t\t\t\t\tName\t\t\t\t\t\t\t \n \n \n \n \n\t\t\t\t\t\t\t\tEmail\t\t\t\t\t\t\t \n \n \n \n \n\t\t\t\t\t\t\t\tPhone Number\t\t\t\t\t\t\t \n \n \n \n \n\t\t\t\t\t\t\t\tCourse Interested\t\t\t\t\t\t\t \n \n \n   \n \n Select \n AWS & DevOps \n Data Science \n MERN Stack \n React Native \n Azure DevOps \n Data Analyst \n SOC Analysis \n Software Developer 

*italicised text*## Load txt


In [34]:
from langchain_community.document_loaders import TextLoader

doc = TextLoader("/content/course.txt")
txt = doc.load()

In [35]:
txt

[Document(metadata={'source': '/content/course.txt'}, page_content='course\tduration\ttiming\ttrainer_name\nData Science\t4 months\t7pm-9pm\tVarun Maya\nAI Specialist\t3 months\t8pm-10pm\tRahul Verma\nData Analytics\t3 months\t6pm-7:30pm\tSneha Reddy\nDigital Marketing\t2 months\t7am-9am\tImran Khan\nSOC analyst\t2 months\t9pm-10:30pm\tPooja Sharma\nBusiness Analyst\t3 months\t5pm-6:30pm\tKiran Patel\n\t\t\t\n\t\t\t')]

## Create Vector Database

In [36]:
all_text = str(txt)+ "\n" + result

from langchain_huggingface import HuggingFaceEmbeddings

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_text(all_text)





In [37]:
chunks

["[Document(metadata={'source': '/content/course.txt'}, page_content='course\\tduration\\ttiming\\ttrainer_name\\nData Science\\t4 months\\t7pm-9pm\\tVarun Maya\\nAI Specialist\\t3 months\\t8pm-10pm\\tRahul Verma\\nData Analytics\\t3 months\\t6pm-7:30pm\\tSneha Reddy\\nDigital Marketing\\t2 months\\t7am-9am\\tImran Khan\\nSOC analyst\\t2 months\\t9pm-10:30pm\\tPooja Sharma\\nBusiness Analyst\\t3 months\\t5pm-6:30pm\\tKiran Patel\\n\\t\\t\\t\\n\\t\\t\\t')]",
 'Full Stack Academy | Gateway To Software Industry \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Full Stack Academy | Gateway To Software Industry \n \n \n \n \n \n \n   \n \n \n \n \n \n \n \n Menu \n \n   \n \n \n Home \n Courses \n Branches \n \n FSA – Tolichowki \n FSA – Gachibowli \n FSA – Ameerpet \n FSA – Charminar \n \n \n About \n Events \n Placements \n \n Companies \n Successfully Placed Candidates \n Candidates available for Placements \n \n \n Contact \n \n \n \n \n \n \n \n 

In [38]:

embeddings = OpenAIEmbeddings(
    model= "text-embedding-3-small"
)


vectorstore = FAISS.from_texts(chunks, embeddings)

In [39]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model="openai/gpt-oss-120b:free",
    temperature=0.7)

In [40]:
from langchain_google_genai import ChatGoogleGenerativeAI


llm1 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key="AIzaSyBK4pTizLkov58ZHPyJCb0b6dstgQc",

    temperature=0.7)

In [41]:
!pip install langchain-classic

In [42]:
#llm + context (all_text) + history (manual retrival flow)

In [43]:


from langchain_classic.chains.retrieval_qa.base import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)


In [44]:
qa_chain

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the user's question.\nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7b99b02f1010>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7b99c93ee0f0>, root_client=<openai.OpenAI object at 0x7b99c9

In [45]:
question = "who is the trainer for data science in full stack academy "
answer = qa_chain.invoke({"query": question})
print(answer['result'])

I’m sorry, but I don’t have that information.


In [46]:
# Ask questions

question = "What is this document about?"
answer = qa_chain.invoke({"query": question})
print(answer['result'])

The document is a promotional overview of the training institute’s course offerings and related information. It includes:

* **A table of core courses** with their duration, class timing, and trainer name (e.g., Data Science – 4 months, 7 pm‑9 pm, trainer Varun Maya; AI Specialist – 3 months, 8 pm‑10 pm, trainer Rahul Verma, etc.).
* **Highlights for two flagship programs** – an “AI‑Powered Web Developer” (MERN‑stack + Claude AI) that runs for 90 days and a “Data Science” program that runs for 4 months. Both show high student satisfaction scores (≈ 4.9/5 and 4.8/5) and mention live projects.
* **Marketing elements** such as curriculum previews, certification badges, and calls to “View all courses.”
* **Additional sections** about the institute’s branches, testimonials, and a list of placed students with their internship or job titles (e.g., Software Developer Intern, Backend Developer Intern, DevOps Engineer, etc.).

Overall, the document serves to inform prospective students about the

In [48]:
!pip install gradio

## Load LLM

## Gradio Chat UI

In [49]:
def chatbot_interface(question):
    answer = qa_chain.invoke({"query": question})
    return answer['result']

In [51]:
iface = gr.Interface(
    fn=chatbot_interface,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about the courses or academy..."),
    outputs="text",
    title="Full Stack Academy Chatbot",
    description="Ask any question related to the courses, trainers, or the academy."
)

iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5658b3bfc9455f36e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
